In [ ]:
# Cell 1: 安装依赖（第一次运行时执行）
# !pip install transformers torch pandas tqdm

In [ ]:
# Cell 2: 导入
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [ ]:
# Cell 3: 从 kg.csv 收集所有唯一实体
df = pd.read_csv('../data/kg.csv')

entities = {}
for _, row in df.iterrows():
    entities[f"{row['x_type']}::{row['x_index']}"] = str(row['x_name']) if pd.notna(row['x_name']) else str(row['x_index'])
    entities[f"{row['y_type']}::{row['y_index']}"] = str(row['y_name']) if pd.notna(row['y_name']) else str(row['y_index'])

print(f'共 {len(entities)} 个实体')

In [ ]:
# Cell 4: 加载 PubMedBERT
MODEL_NAME = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval().to(device)
print(f'使用设备: {device}')

In [ ]:
# Cell 5: 批量编码并保存
BATCH_SIZE = 64  # 显存不足时调小

ids    = list(entities.keys())
names  = list(entities.values())
records = []

with torch.no_grad():
    for i in tqdm(range(0, len(ids), BATCH_SIZE)):
        batch_ids   = ids[i : i + BATCH_SIZE]
        batch_names = names[i : i + BATCH_SIZE]

        inputs = tokenizer(batch_names, padding=True, truncation=True,
                           max_length=64, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # 取 [CLS] token 作为嵌入，与原 BioBERT 用法一致
        embs = model(**inputs).last_hidden_state[:, 0, :].cpu().numpy()

        for eid, emb in zip(batch_ids, embs):
            records.append({'id': eid, 'embedding': emb})

# 保存，格式与原项目 entities_embeddings.pkl 完全一致
pd.DataFrame(records).to_pickle('../data/entities_embeddings.pkl')
print('完成！')